In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark import pipelines as dp

In [0]:
df=spark.read.format("delta")\
    .load("/Volumes/workspace/bronze/bronzevolume/airports/data/")
df=df.drop("_rescued_data")\
        .withColumn('modification',current_timestamp())

display(df)

In [0]:
@dp.view(
    name="trans_airports"
)
def trans_airports():

    df=spark.readStream.format("delta")\
        .load("/Volumes/workspace/bronze/bronzevolume/airports/data")
    return df

In [0]:
dp.create_streaming_table("silver_airports")

dp.create_auto_cdc_flow(
  target = "silver_airports",
  source = "trans_airports",
  keys = ["airport_id"],
  sequence_by = col("airport_id"),
  stored_as_scd_type = 1
)